# wrap seacell functions

In [ ]:
def preprocess(ad, n_top_genes):
    raw_ad = sc.AnnData(ad.X)
    raw_ad.obs_names, raw_ad.var_names = ad.obs_names, ad.var_names
    ad.raw = raw_ad
    # Normalize cells, log transform and compute highly variable genes
    # sc.pp.normalize_per_cell(ad)
    # sc.pp.log1p(ad)
    sc.pp.highly_variable_genes(ad, n_top_genes=n_top_genes)
    # Compute principal components - 
    # Here we use 50 components. This number may also be selected by examining variance explaint
    sc.tl.pca(ad, n_comps=50, use_highly_variable=True)
    return adata

def compute_seacells(ad, n_SEACells, build_kernel_on = 'X_pca'):

  ## Additional parameters
  n_waypoint_eigs = 10 # Number of eigenvalues to consider when initializing metacells

  model = SEACells.core.SEACells(ad,
                  build_kernel_on=build_kernel_on,
                  n_SEACells=n_SEACells,
                  n_waypoint_eigs=n_waypoint_eigs,
                  convergence_epsilon = 1e-5)

  model.construct_kernel_matrix()
  M = model.kernel_matrix
  # Initialize archetypes
  model.initialize_archetypes()
  model.fit(min_iter=10, max_iter=50)

  SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')
  # SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(ad, model.A_, celltype_label='celltype',summarize_layer='raw', minimum_weight=0.05)
  return ad, SEACell_ad, model

def get_metacell_metrics(ad, cell_type_key = 'celltype'):
    if 'X_pca' not in ad.obsm:
        sc.tl.pca(ad)
    purity = SEACells.evaluate.compute_celltype_purity(ad, cell_type_key)
    compactness = SEACells.evaluate.compactness(ad, 'X_pca')
    separation = SEACells.evaluate.separation(ad, 'X_pca',nth_nbr=1)
    return {
    "purity": purity,
    "compactness": compactness,
    "separation": separation
}

In [ ]:
def save_seacell_df(named_dfs, p):
    for name, df in named_dfs.items():
        df.to_csv(f'{p}/{name}.csv', index=False)

In [ ]:
def scproto_metacell_metrics(t, path):
    model = t.load_model()
    proto_ids = t.encode_adata(t.ref.adata, model, True, True)
    adata = t.ref.adata
    adata.obs['SEACell'] = proto_ids.detach().cpu().numpy()
    metacell_metrics = get_metacell_metrics(adata, t.dataset.cell_type_key)
    save_seacell_df(metacell_metrics, path)
    return metacell_metrics

# load and save cd34 adata + finding seacells

In [ ]:
!mkdir data/
!wget https://dp-lab-data-public.s3.amazonaws.com/SEACells-multiome/cd34_multiome_rna.h5ad -O data/cd34_multiome_rna.h5ad 

In [ ]:
import scanpy as sc
# Load the data using 
ad = sc.read('data/cd34_multiome_rna.h5ad')
ad

In [ ]:
# Plot cell-types for reference
sc.pl.scatter(ad, basis='umap', color='celltype', frameon=False)

In [ ]:
preprocess(ad)

In [ ]:
model = find_seacells(ad)

In [ ]:
SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')
SEACell_soft_ad = SEACells.core.summarize_by_soft_SEACell(ad, model.A_, celltype_label='celltype',summarize_layer='raw', minimum_weight=0.05)
SEACell_soft_ad.obs.head()

In [ ]:
SEACells.plot.plot_2D(ad, key='X_umap', colour_metacells=False)

In [ ]:
purity, compactness, separation = get_metacell_metrics(ad)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(4,4))
sns.boxplot(data=purity, y='celltype_purity')
plt.title('Celltype Purity')
sns.despine()
plt.show()
plt.close()

In [ ]:
p = '~/models/cd34/seacell/metrics'


save_seacell_df({
    "purity": purity,
    "compactness": compactness,
    "separation": separation
}, p)

In [ ]:
ls ~/models/cd34/seacell/metrics

# immune 1 ds

In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad('/ictstr01/home/icb/fatemehs.hashemig//data/scpoli/Immune_ALL_human_hvg.h5ad')
adata

In [ ]:
adata.obs.study.value_counts()

In [ ]:
adata_1ds = adata[adata.obs.study == 'Oetjen'].copy()

In [ ]:
sc.tl.pca(adata_1ds, n_comps=50, use_highly_variable=True)


In [ ]:
6881 / 90

In [ ]:
len(adata_1ds)/75

In [ ]:
model = find_seacells(adata_1ds, n_SEACells=128)

In [ ]:
purity, compactness, separation = get_metacell_metrics(adata_1ds, 'final_annotation')

In [ ]:
p = '~/models/pbmc-immune/seacell/metrics'

save_seacell_df({
    "purity": purity,
    "compactness": compactness,
    "separation": separation
}, p)

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(4,4))
sns.boxplot(data=purity, y='celltype_purity')
plt.title('Celltype Purity')
sns.despine()
plt.show()
plt.close()

# immune all

In [ ]:
import scanpy as sc

adata = sc.read_h5ad('/home/icb/fatemehs.hashemig/data/scpoli/Immune_ALL_human.h5ad')

In [ ]:
adata = preprocess(adata, 4000)

In [ ]:
len(adata)//75

In [ ]:
ad, SEACell_ad, model = compute_seacells(adata, 450)
model.plot_convergence()

In [ ]:
from scib_metrics.benchmark import Benchmarker
import os
import numpy as np
def get_scib(adata, emb_keys, batch_key, label_key):
  for emb_key in emb_keys:
      adata.obsm[emb_key] = adata.obsm[emb_key].astype(np.float32)
  bm = Benchmarker(
          adata=adata,
          batch_key=batch_key,
          label_key=label_key,
          embedding_obsm_keys=emb_keys,  # evaluate the PCA space
      )

  bm.benchmark()               # runs neighbors, clustering, and metrics
  results = bm.get_results(min_max_scale=False)   # returns a tidy DataFrame
  return results

def save_scib(results, dataset_name, append=True):
  save_path = f'/ictstr01/home/icb/fatemehs.hashemig/scproto-results/{dataset_name}/'
  os.makedirs(os.path.dirname(save_path), exist_ok=True)
  results = results.drop(index='Metric Type')
  if append:
    saved_res = pd.read_csv(f'{save_path}/scib.csv', index_col = 0)
    results = pd.concat([results, saved_res])
  results.to_csv(f'{save_path}/scib.csv')
  return results

In [ ]:
SEACell_ad

In [ ]:
  SEACell_ad = SEACells.core.summarize_by_SEACell(ad, SEACells_label='SEACell', summarize_layer='raw')

In [ ]:
def add_seacell_lables(label, SEACell_ad, adata):
    # 4. Cell type annotation (most frequent celltype per SEACell)
    SEACell_ad.obs[label] = (
        adata.obs
        .groupby('SEACell')[label]
        .agg(lambda x: x.mode()[0])
        .reindex(SEACell_ad.obs_names)
    )

In [ ]:
add_seacell_lables('final_annotation', SEACell_ad, ad)
add_seacell_lables('study', SEACell_ad, ad)


In [ ]:
def get_seacell_scib(ad, n_hvg):
  # Normalize cells, log transform and compute highly variable genes
  sc.pp.normalize_per_cell(ad)
  sc.pp.log1p(ad)
  sc.pp.highly_variable_genes(ad, n_top_genes=n_hvg)
  sc.tl.pca(ad, n_comps=50, use_highly_variable=True)
  ad.obsm['seacell_pca'] = ad.obsm.pop('X_pca')
  return get_scib(ad, ['seacell_pca'], 'final_annotation', 'study')

res = get_seacell_scib(SEACell_ad, 4000)

In [ ]:
SEACell_ad

In [ ]:
pwd

In [ ]:
SEACell_ad.write('seacell.h5ad')

In [ ]:
get_scib(adata, ['X_pca'], 'final_annotation', 'study')

In [ ]:
imort numpy as np
SEACell_ad.obsm['seacell_pca'] = SEACell_ad.obsm['seacell_pca'].astype(np.float32)

# scProto metacell metrics

In [ ]:
import os
os.chdir('/ictstr01/home/icb/fatemehs.hashemig/codes/interpretable-ssl')

In [ ]:
from interpretable_ssl.trainers.swav import *

params = {
    'cvae_loss_scaler': 0.01,
    'propagation_reg': 1,
    'experiment_name': 't2',
}

## pancreas 1ds

In [ ]:

pancreas = {
    "dataset_id": "pancreas",
    "num_prototypes": 50,
    "study_id": "inDrop3",
    "batch_size": 128
}


t = SwAV(debug = 1, **(params|pancreas))

In [ ]:
t.setup()
t.get_dump_path()

In [ ]:
model = t.load_model()

In [ ]:
# assign each sample to closest proto
proto_ids = t.encode_adata(t.ref.adata, model, True, True)
proto_ids.shape

In [ ]:
adata = t.ref.adata
adata.obs['SEACell'] = proto_ids.detach().cpu().numpy()

In [ ]:
adata

In [ ]:
metacell_metrics = get_metacell_metrics(adata, 'cell_type')

In [ ]:
mkdir ~/models/pancreas/inDrop3-scproto/

In [ ]:
save_seacell_df(metacell_metrics, path)

## immune 1 ds

In [ ]:
immune = {"study_id": "10X", "num_prototypes": 150}


t = SwAV(debug = 1, **(params|immune))

In [ ]:
t.setup()

In [ ]:
mkdir ~/models/pbmc-immune/10X-scproto/

In [ ]:
metrics = scproto_metacell_metrics(t, '~/models/pbmc-immune/10X-scproto/')
metrics

## cd34

In [ ]:
cd34 = {"dataset_id": "cd34", "num_prototypes": 90, "batch_size": 128}

t = SwAV(debug = 1, **(params|cd34))

In [ ]:
t.setup()

In [ ]:
mkdir ~/models/cd34/scproto/

In [ ]:
t.ref.adata

In [ ]:
metrics = scproto_metacell_metrics(t, '~/models/cd34/scproto/', )
metrics